## Действительно ли помогают неразмеченные данные?

Частичное обучение (semi-supervised learning) предлагает методы работы с выборками, в которых лишь для части объектов известны ответы. В статьях утверждается, что добавление неразмеченных данных позволяет повысить качество работы — давайте выясним, так ли это!

Наверное, проще всего добыть неразмеченные примеры, если речь идёт о работе с текстами или изображениями. Остановимся на текстах.

Будем работать с данными из соревнования Predict closed questions on Stack Overflow: https://www.kaggle.com/c/predict-closed-questions-on-stack-overflow/data

Нас будет интересовать файл train-sample.csv — загрузите его. Будем решать бинарную задачу: отнесём объект к классу 1, если `OpenStatus == 'open'`, и к классу 0 иначе.

**Задание 1. (5 баллов)**

Загрузите данные и подготовьте выборку. В качестве признаков возьмите TF-IDF по BodyMarkdown с `min_df=10`; про целевую переменную написано выше. Выделите тестовую выборку из 5000 объектов.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# 1. Загрузка данных
# Предполагается, что файл train-sample.csv находится в текущей папке
df = pd.read_csv('train-sample.csv')

# 2. Формирование целевой переменной
# Класс 1, если OpenStatus == 'open', иначе 0
df['target'] = (df['OpenStatus'] == 'open').astype(int)

# 3. Обработка пропусков в тексте BodyMarkdown (заполняем пустой строкой)
df['BodyMarkdown'] = df['BodyMarkdown'].fillna('')

# 4. Выделение признаков с помощью TF-IDF
# Параметр min_df=10 означает, что термин должен встречаться минимум в 10 документах
vectorizer = TfidfVectorizer(min_df=10)
X = vectorizer.fit_transform(df['BodyMarkdown'])
y = df['target']

# 5. Разделение на обучающую и тестовую выборки (тест — 5000 объектов)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=5000, random_state=42, stratify=y
)

# 6. Вывод информации о размерах полученных выборок
print(f'Размер обучающей выборки: {X_train.shape[0]} объектов, {X_train.shape[1]} признаков')
print(f'Размер тестовой выборки: {X_test.shape[0]} объектов')
print(f'Распределение классов в обучающей выборке:\n{y_train.value_counts()}')
print(f'Распределение классов в тестовой выборке:\n{y_test.value_counts()}')

Размер обучающей выборки: 135272 объектов, 23179 признаков
Размер тестовой выборки: 5000 объектов
Распределение классов в обучающей выборке:
target
0    67636
1    67636
Name: count, dtype: int64
Распределение классов в тестовой выборке:
target
1    2500
0    2500
Name: count, dtype: int64


Нас будут интересовать качество (AUC-ROC) в четырёх следующих постановках:
1. Модель обучается только на размеченных данных.
2. Модель обучается на размеченных и неразмеченных данных, причём неразмеченная часть не пересекается с тестовой выборкой.
3. Модель обучается на размеченных и неразмеченных данных, причём неразмеченная часть совпадает с тестовой выборкой.
4. Модель обучается на размеченных и неразмеченных данных, причём неразмеченная часть включает в себя тестовую выборку.

**Задание 1. (5 баллов)**

Проведите эксперименты и сделайте выводы для любого из методов пакета `sklearn.semi_supervised` и для логистической регрессии

In [2]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.semi_supervised import SelfTrainingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

# Загрузка данных и подготовка признаков (как в задании 1)
df = pd.read_csv('train-sample.csv')
df['target'] = (df['OpenStatus'] == 'open').astype(int)
df['BodyMarkdown'] = df['BodyMarkdown'].fillna('')

vectorizer = TfidfVectorizer(min_df=10)
X = vectorizer.fit_transform(df['BodyMarkdown'])
y = df['target'].values

# Разделяем на тест (5000) и остальное
X_other, X_test, y_other, y_test = train_test_split(
    X, y, test_size=5000, random_state=42, stratify=y
)

# Из остального выделяем размеченную часть (1000) и пул неразмеченных
X_labeled, X_unlabeled_pool, y_labeled, y_unlabeled_pool = train_test_split(
    X_other, y_other, train_size=1000, random_state=42, stratify=y_other
)

# Для сценария 2: неразмеченные не пересекаются с тестом
# Возьмем 5000 объектов из пула (там достаточно)
X_unlabeled_2 = X_unlabeled_pool[:5000]
# Для сценария 3: неразмеченные = тест
X_unlabeled_3 = X_test
# Для сценария 4: неразмеченные включают тест + еще часть из пула
X_unlabeled_4 = np.vstack([X_test.toarray(), X_unlabeled_pool[:5000].toarray()])  # объединяем, но нужно сохранить разреженность? Для простоты преобразуем в плотные, но лучше использовать разреженные
# Для работы с разреженными матрицами используем scipy.sparse.vstack
from scipy.sparse import vstack
X_unlabeled_4 = vstack([X_test, X_unlabeled_pool[:5000]])

# Создаем метки для semi-supervised: для размеченных - истинные, для неразмеченных - -1
y_labeled_true = y_labeled.copy()
y_unlabeled_dummy = np.full(X_unlabeled_2.shape[0], -1)  # для сценария 2

# Базовый классификатор для SelfTrainingClassifier
base_clf = LogisticRegression(max_iter=1000, random_state=42)

# Обучаем логистическую регрессию только на размеченных (baseline)
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_labeled, y_labeled)
lr_pred = lr.predict_proba(X_test)[:, 1]
lr_auc = roc_auc_score(y_test, lr_pred)
print(f'Logistic Regression (only labeled) AUC: {lr_auc:.4f}')

# Сценарий 1: SelfTrainingClassifier только на размеченных (без unlabeled)
# В SelfTrainingClassifier можно передать только размеченные, тогда он работает как базовый классификатор
st1 = SelfTrainingClassifier(base_clf, threshold=0.75, criterion='threshold')
st1.fit(X_labeled, y_labeled)  # здесь все метки известны
st1_pred = st1.predict_proba(X_test)[:, 1]
st1_auc = roc_auc_score(y_test, st1_pred)
print(f'SelfTraining (only labeled) AUC: {st1_auc:.4f}')

# Сценарий 2: SelfTraining с неразмеченными, не пересекающимися с тестом
X_combined_2 = vstack([X_labeled, X_unlabeled_2])
y_combined_2 = np.concatenate([y_labeled, np.full(X_unlabeled_2.shape[0], -1)])
st2 = SelfTrainingClassifier(base_clf, threshold=0.75, criterion='threshold')
st2.fit(X_combined_2, y_combined_2)
st2_pred = st2.predict_proba(X_test)[:, 1]
st2_auc = roc_auc_score(y_test, st2_pred)
print(f'SelfTraining (unlabeled not test) AUC: {st2_auc:.4f}')

# Сценарий 3: SelfTraining с неразмеченными, совпадающими с тестом
X_combined_3 = vstack([X_labeled, X_test])
y_combined_3 = np.concatenate([y_labeled, np.full(X_test.shape[0], -1)])
st3 = SelfTrainingClassifier(base_clf, threshold=0.75, criterion='threshold')
st3.fit(X_combined_3, y_combined_3)
st3_pred = st3.predict_proba(X_test)[:, 1]
st3_auc = roc_auc_score(y_test, st3_pred)
print(f'SelfTraining (unlabeled = test) AUC: {st3_auc:.4f}')

# Сценарий 4: SelfTraining с неразмеченными, включающими тест
X_combined_4 = vstack([X_labeled, X_unlabeled_4])
y_combined_4 = np.concatenate([y_labeled, np.full(X_unlabeled_4.shape[0], -1)])
st4 = SelfTrainingClassifier(base_clf, threshold=0.75, criterion='threshold')
st4.fit(X_combined_4, y_combined_4)
st4_pred = st4.predict_proba(X_test)[:, 1]
st4_auc = roc_auc_score(y_test, st4_pred)
print(f'SelfTraining (unlabeled includes test) AUC: {st4_auc:.4f}')

Logistic Regression (only labeled) AUC: 0.7579
SelfTraining (only labeled) AUC: 0.7579


/usr/local/lib/python3.12/dist-packages/sklearn/semi_supervised/_self_training.py:288: UserWarning: y contains no unlabeled samples
  warnings.warn("y contains no unlabeled samples", UserWarning)


SelfTraining (unlabeled not test) AUC: 0.7474
SelfTraining (unlabeled = test) AUC: 0.7481
SelfTraining (unlabeled includes test) AUC: 0.7418


### Self-train

Обучаем на размеченной части, предсказываем неразмеченную, потом обучаем на всех, и предсказываем неразмеченную, повторяем пока не сойдёмся в предскзааниях неразмеченной части

In [3]:
import numpy as np
from sklearn.base import clone

def self_train(X_labeled, y_labeled, X_unlabeled, base_estimator,
               threshold=0.75, max_iter=100, verbose=True):
    """
    Реализация алгоритма self-training.

    Параметры:
    ----------
    X_labeled : array-like, shape (n_labeled, n_features)
        Признаки размеченных объектов.
    y_labeled : array-like, shape (n_labeled,)
        Метки размеченных объектов.
    X_unlabeled : array-like, shape (n_unlabeled, n_features)
        Признаки неразмеченных объектов.
    base_estimator : объект с методами fit и predict_proba
        Базовый классификатор, который будет дообучаться.
    threshold : float, default=0.75
        Порог уверенности для добавления псевдо-меток.
    max_iter : int, default=100
        Максимальное число итераций.
    verbose : bool, default=True
        Если True, печатать информацию о ходе выполнения.

    Возвращает:
    -----------
    model : обученный классификатор (экземпляр base_estimator)
    history : list
        Список количества размеченных объектов после каждой итерации.
    """
    # Клонируем базовую модель, чтобы не изменять исходную
    model = clone(base_estimator)

    # Текущие размеченные данные (со временем будем расширять)
    X_train = np.array(X_labeled)
    y_train = np.array(y_labeled)

    # Неразмеченные данные (копируем, чтобы не менять исходный массив)
    X_unlabeled = np.array(X_unlabeled)

    # Массив для хранения текущих псевдо-меток (изначально None)
    # Будем использовать для проверки сходимости
    prev_pseudo_labels = None

    history = [len(X_train)]

    for iteration in range(max_iter):
        # Обучаем модель на текущем размеченном наборе
        model.fit(X_train, y_train)

        # Получаем вероятности для неразмеченных объектов
        proba = model.predict_proba(X_unlabeled)
        # Уверенность = максимум вероятности по классам
        confidence = np.max(proba, axis=1)
        # Предсказанные классы
        pred_classes = model.classes_[np.argmax(proba, axis=1)]

        # Выбираем объекты, уверенность выше порога
        high_conf_mask = confidence >= threshold
        new_X = X_unlabeled[high_conf_mask]
        new_y = pred_classes[high_conf_mask]

        # Если нет новых объектов для добавления — останавливаемся
        if len(new_X) == 0:
            if verbose:
                print(f"Итерация {iteration+1}: нет объектов выше порога. Остановка.")
            break

        # Проверяем сходимость по псевдо-меткам
        # Для этого сравним новые метки с предыдущими (если они были)
        if prev_pseudo_labels is not None:
            # Получаем метки для тех же объектов, которые уже были размечены ранее
            # Но проще сравнить метки только для добавляемых объектов? Или для всех?
            # Здесь для простоты будем считать, что сходимость наступила, если добавляемые объекты
            # уже были добавлены ранее с теми же метками. Но более строгий подход — сравнение
            # всех псевдо-меток на неразмеченном множестве.
            # Реализуем через сравнение предсказаний для всего неразмеченного множества
            # с предыдущей итерацией.
            current_labels = np.full(X_unlabeled.shape[0], -1)
            current_labels[high_conf_mask] = new_y
            if prev_pseudo_labels is not None and np.array_equal(prev_pseudo_labels, current_labels):
                if verbose:
                    print(f"Итерация {iteration+1}: псевдо-метки перестали меняться. Остановка.")
                break
            prev_pseudo_labels = current_labels
        else:
            # Инициализируем массив для предыдущих меток
            prev_pseudo_labels = np.full(X_unlabeled.shape[0], -1)
            prev_pseudo_labels[high_conf_mask] = new_y

        # Добавляем новые объекты в обучающий набор
        X_train = np.vstack([X_train, new_X])
        y_train = np.concatenate([y_train, new_y])

        # Удаляем добавленные объекты из неразмеченного множества
        X_unlabeled = X_unlabeled[~high_conf_mask]

        history.append(len(X_train))

        if verbose:
            print(f"Итерация {iteration+1}: добавлено {len(new_X)} объектов, "
                  f"всего размечено {len(X_train)}")

    # Финальное обучение на всех накопленных данных (можно не делать, если модель уже обучена на последней итерации)
    # Но для гарантии обучим ещё раз
    model.fit(X_train, y_train)

    return model, history

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

# Генерация синтетических данных
X, y = make_classification(n_samples=5000, n_features=20, random_state=42)
X_labeled, X_unlabeled, y_labeled, _ = train_test_split(X, y, train_size=200, random_state=42)

# Базовый классификатор
base_lr = LogisticRegression(max_iter=1000)

# Запуск self-training
model, history = self_train(X_labeled, y_labeled, X_unlabeled, base_lr,
                            threshold=0.8, max_iter=20, verbose=True)

print("История размеченных объектов:", history)

Итерация 1: добавлено 3438 объектов, всего размечено 3638
Итерация 2: добавлено 751 объектов, всего размечено 4389
Итерация 3: добавлено 176 объектов, всего размечено 4565
Итерация 4: добавлено 38 объектов, всего размечено 4603
Итерация 5: добавлено 19 объектов, всего размечено 4622
Итерация 6: добавлено 13 объектов, всего размечено 4635
Итерация 7: добавлено 11 объектов, всего размечено 4646
Итерация 8: добавлено 12 объектов, всего размечено 4658
Итерация 9: добавлено 6 объектов, всего размечено 4664
Итерация 10: добавлено 3 объектов, всего размечено 4667
Итерация 11: добавлено 4 объектов, всего размечено 4671
Итерация 12: добавлено 1 объектов, всего размечено 4672
Итерация 13: добавлено 1 объектов, всего размечено 4673
Итерация 14: нет объектов выше порога. Остановка.
История размеченных объектов: [200, 3638, 4389, 4565, 4603, 4622, 4635, 4646, 4658, 4664, 4667, 4671, 4672, 4673]
